# Lab: Gibbs Sampling for Hierarchical Gamma-Poisson — and the Power of Reparametrization

**Data 145, Spring 2026: Evidence and Uncertainty**

---

In Lecture 26 we ran a Gibbs sampler for the Premier League goal-scoring model with the shape $\alpha$ held *fixed*. In a real analysis, we'd want to learn $\alpha$ from the data too. This lab shows what happens when we sample both $\alpha$ and $\beta$, and how a clever reparametrization dramatically improves mixing.

**Outline:**

1. **Recap of the model** and review of Gibbs sampling.
2. **Conjugate updates** for $\theta$ and $\beta$ — these are easy.
3. **Updating $\alpha$ numerically** — no conjugacy, but the conditional is one-dimensional.
4. **Run the $(\alpha, \beta)$ Gibbs sampler** — observe slow, ridge-bound mixing.
5. **Reparametrize to $(\mu, \beta)$**, where $\mu = \alpha/\beta$ is the population mean.
6. **Run the $(\mu, \beta)$ sampler** — much faster mixing.
7. **Compare the two samplers** and explain why reparametrization helps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.special import gammaln
from IPython.display import HTML

plt.rcParams.update({
    'figure.figsize': (10, 4),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

COLOR_AB     = '#648FFF'   # (alpha, beta) parametrization — blue
COLOR_MUBETA = '#DC267F'   # (mu, beta) parametrization — magenta
COLOR_DATA   = 'black'

rng = np.random.default_rng(0)

---

## 1. The Premier League data and the model

We use the same data as in Worksheet 4 and Lecture 26: total goals scored per match by each of the 20 teams in the first 10 matches of the 2023-24 Premier League season.

**Model.** For team $i = 1, \ldots, d$ and match $j = 1, \ldots, m$:
$$\theta_i \mid \alpha, \beta \stackrel{iid}{\sim} \text{Gamma}(\alpha, \beta), \qquad X_{ij} \mid \theta_i \sim \text{Poisson}(\theta_i).$$

The unknowns are:
- $\theta_i$ — team $i$'s underlying scoring rate ($d = 20$ of them),
- $\alpha, \beta$ — population-level hyperparameters.

We put diffuse but proper priors $\alpha, \beta \sim \text{Gamma}(1, 0.05)$.

In [ ]:
teams = ['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton',
         'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham',
         'Liverpool', 'Luton', 'Man City', 'Man United', 'Newcastle',
         "Nott'm Forest", 'Sheffield United', 'Tottenham', 'West Ham', 'Wolves']

goals = np.array([
    [2, 2, 3, 3, 6, 3, 0, 4, 3, 1],  # Arsenal
    [3, 5, 1, 0, 2, 0, 0, 3, 1, 1],  # Aston Villa
    [1, 1, 2, 2, 2, 1, 0, 1, 1, 2],  # Bournemouth
    [1, 2, 2, 3, 0, 0, 3, 1, 3, 1],  # Brentford
    [0, 0, 3, 4, 0, 2, 1, 1, 0, 2],  # Brighton
    [1, 2, 5, 2, 0, 2, 1, 0, 0, 0],  # Burnley
    [2, 2, 0, 2, 3, 2, 4, 5, 4, 1],  # Chelsea
    [1, 0, 1, 3, 1, 2, 2, 4, 0, 0],  # Crystal Palace
    [1, 1, 2, 1, 2, 1, 1, 1, 3, 3],  # Everton
    [3, 1, 1, 0, 2, 3, 0, 0, 0, 5],  # Fulham
    [4, 1, 3, 4, 1, 3, 4, 1, 2, 2],  # Liverpool
    [1, 2, 1, 4, 0, 1, 1, 1, 3, 2],  # Luton
    [5, 3, 4, 3, 5, 6, 3, 4, 0, 0],  # Man City
    [4, 0, 1, 1, 3, 3, 1, 0, 2, 2],  # Man United
    [2, 3, 1, 1, 1, 4, 4, 1, 1, 0],  # Newcastle
    [1, 0, 3, 1, 0, 1, 3, 2, 0, 1],  # Nott'm Forest
    [2, 0, 0, 0, 1, 1, 2, 0, 2, 1],  # Sheffield United
    [3, 1, 0, 5, 2, 3, 2, 1, 3, 1],  # Tottenham
    [1, 2, 0, 3, 1, 1, 0, 2, 0, 2],  # West Ham
    [3, 0, 1, 1, 2, 4, 1, 1, 1, 1],  # Wolves
])

d, m = goals.shape
G_team = goals.sum(axis=1)              # total goals per team

# Diffuse hyperprior shared by alpha, beta
c_prior, r_prior = 1.0, 0.05            # Gamma(1, 0.05): mean 20

print(f"d = {d} teams, m = {m} matches each")
print(f"per-team totals (G_i): {G_team}")
print(f"sample mean rate per match: {G_team.mean()/m:.3f}")

---

## 2. Gibbs sampling: a refresher

To sample from the joint posterior $p(\theta, \alpha, \beta \mid X)$ we cycle through the **full conditionals** — the distribution of one block of variables given everything else fixed:

1. Sample $\theta \sim p(\theta \mid \alpha, \beta, X)$.
2. Sample $\beta \sim p(\beta \mid \alpha, \theta, X) = p(\beta \mid \alpha, \theta)$ *(doesn't depend on $X$ given $\theta$)*.
3. Sample $\alpha \sim p(\alpha \mid \beta, \theta, X) = p(\alpha \mid \beta, \theta)$.

Repeat. After enough iterations (burn-in), the draws are approximately from the joint posterior.

**Key observation.** The first two updates are *conjugate* — closed-form Gamma distributions. The third has no conjugate prior, but it's still a one-dimensional density, so we can sample it numerically.

---

## 3. Conjugate updates: $\theta$ and $\beta$

### 3.1 Update for $\theta_i$

Given $\alpha, \beta$ and the data, the team rates $\theta_i$ are conditionally independent and Gamma-Poisson conjugate:

$$\theta_i \mid \alpha, \beta, X \;\sim\; \text{Gamma}\!\bigl(\alpha + G_i,\ \beta + m\bigr).$$

This is exactly the conjugate update from Worksheet 4.

### 3.2 Update for $\beta$

Given $\alpha$ and the team rates $\theta_i$, the prior on $\beta$ (Gamma) is conjugate to the Gamma likelihood for the $\theta_i$'s. Treating $\theta_1, \ldots, \theta_d$ as *data* and $\beta$ as the unknown:

$$p(\beta \mid \alpha, \theta) \;\propto\; \underbrace{\beta^{c-1} e^{-r\beta}}_{\text{prior}} \cdot \underbrace{\prod_{i=1}^d \frac{\beta^\alpha}{\Gamma(\alpha)} \theta_i^{\alpha-1} e^{-\beta\theta_i}}_{\text{likelihood}} \;\propto\; \beta^{c + d\alpha - 1} e^{-(r + \sum\theta_i)\beta},$$

so

$$\beta \mid \alpha, \theta \;\sim\; \text{Gamma}\!\bigl(c + d\alpha,\ r + \textstyle\sum_i \theta_i\bigr).$$

Both updates are one-liners.

In [ ]:
def update_theta(alpha, beta, rng):
    """Sample theta_i | alpha, beta, X. Conjugate Gamma posterior."""
    return rng.gamma(alpha + G_team, scale=1 / (beta + m))

def update_beta(alpha, theta, rng):
    """Sample beta | alpha, theta. Conjugate Gamma posterior."""
    return rng.gamma(c_prior + d * alpha, scale=1 / (r_prior + theta.sum()))

---

## 4. The $\alpha$ update: a one-dimensional density we can compute

There is no conjugate prior for $\alpha$ in this model. But the full conditional of $\alpha$ is just a *one-dimensional* probability density, so we can:

1. Write down a function proportional to $p(\alpha \mid \beta, \theta)$,
2. Evaluate it on a fine grid,
3. Normalize numerically, and
4. Sample from the resulting discrete approximation.

### 4.1 The conditional density

Combining the $\text{Gamma}(c, r)$ prior with the Gamma likelihood:

$$p(\alpha \mid \beta, \theta) \;\propto\; \alpha^{c-1} e^{-r\alpha} \cdot \prod_{i=1}^d \frac{\beta^\alpha}{\Gamma(\alpha)} \theta_i^{\alpha - 1}.$$

Taking logs and dropping terms that don't depend on $\alpha$:

$$\log p(\alpha \mid \beta, \theta) \;=\; (c - 1)\log\alpha - r\alpha \;+\; \alpha\Bigl(d\log\beta + \textstyle\sum_i \log\theta_i\Bigr) \;-\; d\log\Gamma(\alpha) \;+\; \text{const}.$$

This is a *closed-form* function of $\alpha$ that we can evaluate at any $\alpha > 0$.

### 4.2 Grid-based sampling

We evaluate $\log p(\alpha \mid \beta, \theta)$ on a fine grid, subtract the maximum (for numerical stability), exponentiate, and normalize.

In [ ]:
alpha_grid = np.linspace(0.5, 200, 1600)

def log_cond_alpha(alpha, beta, theta):
    """log p(alpha | beta, theta) up to an additive constant."""
    sum_log_theta = np.log(theta).sum()
    return ((c_prior - 1) * np.log(alpha) - r_prior * alpha
            + alpha * (d * np.log(beta) + sum_log_theta)
            - d * gammaln(alpha))

def update_alpha(beta, theta, rng):
    """Sample alpha | beta, theta by grid-based numerical inverse CDF."""
    log_p = log_cond_alpha(alpha_grid, beta, theta)
    log_p -= log_p.max()
    p = np.exp(log_p)
    p /= p.sum()
    return rng.choice(alpha_grid, p=p)

### 4.3 How does the conditional density change as $(\beta, \theta)$ vary?

The conditional $p(\alpha \mid \beta, \theta)$ depends on the current state of $\beta$ and $\theta$, so it shifts from iteration to iteration. To see this, we run a short warm-up chain, pick three iterations whose $\beta$ values span the posterior range (low, middle, high), and overlay the conditional densities of $\alpha$ for each.

In [ ]:
# Run a short warm-up to collect a few (beta, theta) states spanning the posterior
n_warm = 500
beta_warm_trace = np.zeros(n_warm)
theta_warm_trace = np.zeros((n_warm, d))
alpha_w, beta_w = 5.0, 5.0
rng_warm = np.random.default_rng(7)
for t in range(n_warm):
    theta_w = update_theta(alpha_w, beta_w, rng_warm)
    beta_w  = update_beta(alpha_w, theta_w, rng_warm)
    alpha_w = update_alpha(beta_w, theta_w, rng_warm)
    beta_warm_trace[t]  = beta_w
    theta_warm_trace[t] = theta_w

# Pick three iterations (after a small burn-in) spanning low / middle / high beta
post_burn = 100
beta_post  = beta_warm_trace[post_burn:]
theta_post = theta_warm_trace[post_burn:]
order = np.argsort(beta_post)
n_post = len(order)
picks = [order[n_post // 10], order[n_post // 2], order[9 * n_post // 10]]
labels = ['low $\\beta$', 'middle $\\beta$', 'high $\\beta$']
colors = ['#648FFF', '#785EF0', '#DC267F']

fig, ax = plt.subplots(figsize=(11, 5))
for idx, label, color in zip(picks, labels, colors):
    beta_iter  = beta_post[idx]
    theta_iter = theta_post[idx]
    log_p = log_cond_alpha(alpha_grid, beta_iter, theta_iter)
    log_p -= log_p.max()
    p = np.exp(log_p); p /= p.sum() * (alpha_grid[1] - alpha_grid[0])
    leg = fr'{label}: $\beta = {beta_iter:.2f}$, $\sum_i \theta_i = {theta_iter.sum():.1f}$'
    ax.plot(alpha_grid, p, color=color, linewidth=2, label=leg)
    ax.fill_between(alpha_grid, 0, p, color=color, alpha=0.18)

ax.set_xlabel(r'$\alpha$')
ax.set_ylabel(r'$p(\alpha \mid \beta, \theta)$')
ax.set_title(r'Conditional density of $\alpha$ at three iterations with different $\beta$')
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim(0, 50)
plt.tight_layout()
plt.show()

*Each curve is the full conditional of $\alpha$ given the $(\beta, \theta)$ at one iteration of the warm-up chain. As $\beta$ moves from low to high, the conditional shifts to track $\alpha/\beta \approx \hat\mu \approx 1.7$ — that's the ridge that the chain's collective draws hug. **Notice how narrow each conditional is**: at any fixed $\beta$, the new $\alpha$ can only land in a small window. That's the reason the chain takes tiny steps and mixes slowly.*

---

## 5. The full $(\alpha, \beta)$ Gibbs sampler

We now run the full sampler: cycle through $\theta$, $\beta$, $\alpha$ at each iteration.

In [ ]:
def run_alphabeta_sampler(n_iter, alpha0=5.0, beta0=5.0, seed=42):
    """Run Gibbs and record sub-states after EACH conditional update.

    Sub-state layout (length 2*n_iter + 1):
      sub[0]      = (alpha0, beta0)                       initial
      sub[2t+1]   = (alpha, beta) after the beta update   vertical move
      sub[2t+2]   = (alpha, beta) after the alpha update  horizontal move

    For analysis, the post-full-cycle values are sub[2::2] (length n_iter).
    """
    rng_run = np.random.default_rng(seed)
    L = 2 * n_iter + 1
    alpha_sub = np.zeros(L)
    beta_sub  = np.zeros(L)
    theta_trace = np.zeros((n_iter, d))
    alpha, beta = alpha0, beta0
    alpha_sub[0], beta_sub[0] = alpha, beta
    for t in range(n_iter):
        theta = update_theta(alpha, beta, rng_run)
        theta_trace[t] = theta
        beta  = update_beta(alpha, theta, rng_run)        # vertical move
        alpha_sub[2*t+1], beta_sub[2*t+1] = alpha, beta
        alpha = update_alpha(beta, theta, rng_run)        # horizontal move
        alpha_sub[2*t+2], beta_sub[2*t+2] = alpha, beta
    return alpha_sub, beta_sub, theta_trace

n_iter = 5000
alpha_sub, beta_sub, theta_tr = run_alphabeta_sampler(n_iter)
# Post-full-cycle traces, used for the analysis plots and ESS calculations.
alpha_tr = alpha_sub[2::2]
beta_tr  = beta_sub[2::2]

burn = 500
print(f"Posterior mean alpha:  {alpha_tr[burn:].mean():5.2f}  (SD {alpha_tr[burn:].std():.2f})")
print(f"Posterior mean beta:   {beta_tr[burn:].mean():5.2f}  (SD {beta_tr[burn:].std():.2f})")
print(f"Posterior mean alpha/beta:  {(alpha_tr[burn:]/beta_tr[burn:]).mean():.3f}")

### 5.1 The 2D trace of $(\alpha, \beta)$

Each Gibbs step is a horizontal move (update $\beta$ at fixed $\alpha$) followed by a vertical move (update $\alpha$ at fixed $\beta$). If $\alpha$ and $\beta$ are correlated in the posterior, these alternating moves trace out a long, thin ridge.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Marginal traces (log scale on the value axis -- alpha and beta differ in magnitude)
axes[0].plot(alpha_tr, color=COLOR_AB, linewidth=0.5, label=r'$\alpha$')
axes[0].plot(beta_tr,  color='gray',    linewidth=0.5, label=r'$\beta$')
axes[0].set_yscale('log')
axes[0].set_xlabel('iteration')
axes[0].set_ylabel('value (log scale)')
axes[0].set_title(r'Marginal traces of $\alpha$ and $\beta$')
axes[0].legend()

# 2D scatter showing the ridge
axes[1].plot(alpha_tr[burn:], beta_tr[burn:],
             color=COLOR_AB, linewidth=0.3, alpha=0.5)
axes[1].scatter(alpha_tr[burn:], beta_tr[burn:],
                color=COLOR_AB, s=4, alpha=0.4)
axes[1].set_xlabel(r'$\alpha$')
axes[1].set_ylabel(r'$\beta$')
axes[1].set_title(r'Trace in $(\alpha, \beta)$ space')

plt.tight_layout()
plt.show()

*The chain hugs a long thin ridge along which $\alpha/\beta \approx 1.7$ (the population mean rate). Each Gibbs step alternates horizontal and vertical, but because $\alpha$ and $\beta$ are highly correlated, each individual step is small relative to the length of the ridge — the chain mixes slowly along it.*

### 5.2 Animation: watching each conditional update

The static scatter doesn't fully convey *how* the chain explores so slowly. Each Gibbs iteration is two moves: a **vertical** move (update $\beta$ at fixed $\alpha$) and a **horizontal** move (update $\alpha$ at fixed $\beta$). The animation shows each move individually:

- The first **20 sub-steps** play at **5 sub-steps per second** so you can see each $\beta$ and $\alpha$ update.
- After that, the chain plays at **5× speed** to show the longer-range meandering along the ridge.

Watch how the alternating axis-aligned moves stay confined to a thin diagonal band — that's exactly why mixing is slow.

In [ ]:
# Build animation showing each conditional sub-step (vertical/horizontal moves).
# Display rate: 25 fps (interval=40ms).
# Slow phase: each of the first 20 sub-steps held for 5 frames -> 5 sub-steps/sec.
# Fast phase: chain advances 5 sub-steps per frame -> 25 sub-steps/sec (5x faster).

n_slow = 20
slow_frames_per_step = 5
fast_steps_per_frame = 5
n_fast_frames = 250

frame_to_idx = []
# Slow phase: hold each new sub-state for 5 frames
for k in range(1, n_slow + 1):
    for _ in range(slow_frames_per_step):
        frame_to_idx.append(k)
# Fast phase: advance 5 sub-states per frame
current = n_slow
max_idx = len(alpha_sub) - 1
for _ in range(n_fast_frames):
    current = min(current + fast_steps_per_frame, max_idx)
    frame_to_idx.append(current)
    if current == max_idx:
        break

n_frames_anim = len(frame_to_idx)
last_idx = frame_to_idx[-1]

fig_a, ax_a = plt.subplots(figsize=(7, 6))
ax_a.set_xlim(alpha_sub[:last_idx + 1].min() - 1.5, alpha_sub[:last_idx + 1].max() + 1.5)
ax_a.set_ylim(beta_sub[:last_idx + 1].min()  - 1.0, beta_sub[:last_idx + 1].max()  + 1.0)
ax_a.set_xlabel(r'$\alpha$')
ax_a.set_ylabel(r'$\beta$')
ax_a.set_title(r'Gibbs in $(\alpha, \beta)$: alternating $\beta$ (vertical) and $\alpha$ (horizontal) updates')

line, = ax_a.plot([], [], color=COLOR_AB, linewidth=0.7, alpha=0.6)
head, = ax_a.plot([], [], 'o', color='black', markersize=10)
text  = ax_a.text(0.02, 0.97, '', transform=ax_a.transAxes, va='top', fontsize=11,
                  bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

slow_frames_total = n_slow * slow_frames_per_step

def init_a():
    line.set_data([], [])
    head.set_data([], [])
    text.set_text('')
    return line, head, text

def update_a(k):
    end = frame_to_idx[k]
    line.set_data(alpha_sub[:end + 1], beta_sub[:end + 1])
    head.set_data([alpha_sub[end]], [beta_sub[end]])
    iter_num = (end + 1) // 2
    if end == 0:
        ut = 'initial state'
    elif end % 2 == 1:
        ut = r'just updated $\beta$ (vertical)'
    else:
        ut = r'just updated $\alpha$ (horizontal)'
    speed = 'slow' if k < slow_frames_total else '5x speed'
    text.set_text(f'iter {iter_num}: {ut}  [{speed}]')
    return line, head, text

anim_ab = animation.FuncAnimation(
    fig_a, update_a, frames=n_frames_anim,
    init_func=init_a, blit=True, interval=40
)
plt.close(fig_a)
HTML(anim_ab.to_jshtml())

---

## 6. Reparametrization: $(\mu, \beta)$

A $\text{Gamma}(\alpha, \beta)$ has mean $\alpha/\beta$. Define

$$\mu \;=\; \frac{\alpha}{\beta}.$$

The map $(\alpha, \beta) \to (\mu, \beta)$ is one-to-one with inverse $\alpha = \mu\beta$.

**Why might $(\mu, \beta)$ behave better than $(\alpha, \beta)$?**

- $\mu$ is the *population mean* of team rates, which the data pin down very tightly: $\hat\mu \approx \bar X / m$, the overall sample rate per match.
- Given $\mu$, the spread of the $\theta_i$'s around their mean is informative about $\beta$ separately. (Recall $\text{Var}(\theta) = \mu/\beta$ for our Gamma — knowing $\mu$, the spread tells us $\beta$.)
- Empirically, **$\mu$ and $\beta$ are essentially uncorrelated in the posterior** — they answer different questions about the population, and one doesn't constrain the other once we condition on the data.

In contrast, $\alpha$ and $\beta$ are highly correlated: as the data tell us about $\mu$, it constrains $\alpha/\beta$ but leaves $(\alpha, \beta)$ free along the ridge $\alpha = \mu\beta$. We saw this in §5.

**Same prior, different coordinates.** To make the comparison apples-to-apples, we use the *same* prior as in the $(\alpha, \beta)$ sampler — independent $\text{Gamma}(c, r)$ priors on $\alpha$ and $\beta$ — just expressed in $(\mu, \beta)$ coordinates. The change-of-variable formula gives

$$p_{\mu, \beta}(\mu, \beta) \;=\; p_{\alpha, \beta}(\mu\beta,\ \beta) \cdot \left|\det \tfrac{\partial(\alpha, \beta)}{\partial(\mu, \beta)}\right|, \qquad \left|\det \tfrac{\partial(\alpha, \beta)}{\partial(\mu, \beta)}\right| = \beta.$$

Taking logs:

$$\log p_{\mu, \beta}(\mu, \beta) \;=\; (c - 1)\log\mu \;+\; (2c - 1) \log\beta \;-\; r\beta\,(1 + \mu) \;+\; \text{const}.$$

(For $c = 1, r = 0.05$: simplifies to $\log\beta - 0.05\beta(1 + \mu)$.) This is the prior we use in the conditional log-densities below.

---

## 7. Gibbs in $(\mu, \beta)$ parametrization

The $\theta_i$ update is the same as before — just compute $\alpha = \mu\beta$ to use the Gamma-Poisson conjugacy:

$$\theta_i \mid \mu, \beta, X \;\sim\; \text{Gamma}\!\bigl(\alpha + G_i,\ \beta + m\bigr), \quad \text{where } \alpha = \mu\beta.$$

For $\mu$ and $\beta$, neither has a conjugate full conditional in this parametrization (the transformed prior breaks the Gamma-Gamma conjugacy of §3.2). But each conditional is one-dimensional, so we use the same numerical grid trick as for $\alpha$ in §4.

### 7.1 Conditional density of $\mu$

With $\alpha = \mu\beta$, the joint log-likelihood of $\theta_1, \ldots, \theta_d$ given $(\mu, \beta)$ is

$$\sum_i \log p(\theta_i \mid \mu, \beta) \;=\; d\,\mu\beta\,\log\beta - d\log\Gamma(\mu\beta) + (\mu\beta - 1)\sum_i \log\theta_i - \beta\sum_i \theta_i.$$

Add the prior $\log p_{\mu, \beta}(\mu, \beta)$ from §6, and you have a closed-form 1D function of $\mu$. Evaluate on a grid, normalize, sample.

### 7.2 Conditional density of $\beta$

Same expression — vary $\beta$ instead of $\mu$ — combined with the same prior.

In [ ]:
# The (mu, beta) sampler uses the SAME prior as the (alpha, beta) sampler --
# independent Gamma(c_prior, r_prior) on alpha and beta -- transformed to (mu, beta)
# via the Jacobian |d(alpha, beta)/d(mu, beta)| = beta.

mu_grid   = np.linspace(0.05, 5.0,  600)
beta_grid = np.linspace(0.5,  200.0, 1600)

def log_post_mu_beta(mu, beta, theta):
    """log p(mu, beta | theta) up to additive const, using the (alpha, beta) prior
    transformed via Jacobian. Vectorized over mu *or* beta (whichever is array;
    the other should be scalar)."""
    alpha = mu * beta
    sum_log_theta = np.log(theta).sum()
    sum_theta     = theta.sum()
    # Prior in (mu, beta) = prior in (alpha, beta) at (mu*beta, beta) times Jacobian beta.
    # log of that, with independent Gamma(c, r) priors on alpha and beta:
    #   = (c-1) log mu + (2c-1) log beta - r beta (1 + mu)
    log_prior = ((c_prior - 1) * np.log(mu)
                 + (2 * c_prior - 1) * np.log(beta)
                 - r_prior * beta * (1 + mu))
    log_lik = (d * alpha * np.log(beta) - d * gammaln(alpha)
               + (alpha - 1) * sum_log_theta - beta * sum_theta)
    return log_prior + log_lik

def update_mu(beta, theta, rng):
    log_p = log_post_mu_beta(mu_grid, beta, theta)
    log_p -= log_p.max()
    p = np.exp(log_p); p /= p.sum()
    return rng.choice(mu_grid, p=p)

def update_beta_mb(mu, theta, rng):
    """Update beta in the (mu, beta) parametrization. NOT conjugate (the
    transformed prior breaks the conjugacy from sec 3.2), so we use the
    same 1D grid trick."""
    log_p = log_post_mu_beta(mu, beta_grid, theta)
    log_p -= log_p.max()
    p = np.exp(log_p); p /= p.sum()
    return rng.choice(beta_grid, p=p)

In [ ]:
def run_mubeta_sampler(n_iter, mu0=2.0, beta0=10.0, seed=42):
    """Run Gibbs and record sub-states after EACH conditional update.

    Sub-state layout (length 2*n_iter + 1):
      sub[0]      = (mu0, beta0)                            initial
      sub[2t+1]   = (mu, beta) after the mu update          horizontal move
      sub[2t+2]   = (mu, beta) after the beta update        vertical move
    """
    rng_run = np.random.default_rng(seed)
    L = 2 * n_iter + 1
    mu_sub_arr   = np.zeros(L)
    beta_sub_arr = np.zeros(L)
    theta_trace = np.zeros((n_iter, d))
    mu, beta = mu0, beta0
    mu_sub_arr[0], beta_sub_arr[0] = mu, beta
    for t in range(n_iter):
        alpha = mu * beta
        theta = update_theta(alpha, beta, rng_run)
        theta_trace[t] = theta
        mu = update_mu(beta, theta, rng_run)              # horizontal move
        mu_sub_arr[2*t+1], beta_sub_arr[2*t+1] = mu, beta
        beta = update_beta_mb(mu, theta, rng_run)         # vertical move
        mu_sub_arr[2*t+2], beta_sub_arr[2*t+2] = mu, beta
    return mu_sub_arr, beta_sub_arr, theta_trace

mu_sub, beta_mb_sub, theta_tr_mb = run_mubeta_sampler(n_iter)
mu_tr      = mu_sub[2::2]
beta_mb_tr = beta_mb_sub[2::2]

# Convert to (alpha, beta) for direct comparison
alpha_from_mubeta = mu_tr * beta_mb_tr

print(f"Posterior mean mu:    {mu_tr[burn:].mean():.3f}  (SD {mu_tr[burn:].std():.3f})")
print(f"Posterior mean beta:  {beta_mb_tr[burn:].mean():.3f}  (SD {beta_mb_tr[burn:].std():.3f})")
print(f"Posterior mean alpha (from mu*beta):  {alpha_from_mubeta[burn:].mean():.2f}")

### 7.3 The 2D trace of $(\mu, \beta)$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(mu_tr,      color=COLOR_MUBETA, linewidth=0.5, label=r'$\mu$')
axes[0].plot(beta_mb_tr, color='gray',        linewidth=0.5, label=r'$\beta$')
axes[0].set_yscale('log')
axes[0].set_xlabel('iteration')
axes[0].set_ylabel('value (log scale)')
axes[0].set_title(r'Marginal traces of $\mu$ and $\beta$')
axes[0].legend()

axes[1].plot(mu_tr[burn:], beta_mb_tr[burn:],
             color=COLOR_MUBETA, linewidth=0.3, alpha=0.5)
axes[1].scatter(mu_tr[burn:], beta_mb_tr[burn:],
                color=COLOR_MUBETA, s=4, alpha=0.4)
axes[1].set_xlabel(r'$\mu$')
axes[1].set_ylabel(r'$\beta$')
axes[1].set_title(r'Trace in $(\mu, \beta)$ space')

plt.tight_layout()
plt.show()

*A roundish blob, not a thin ridge — and the marginal traces look like white noise rather than slow wandering. The chain explores the posterior efficiently. The empirical correlation between $\mu$ and $\beta$ in the posterior is essentially zero.*

### 7.4 Animation: each conditional update in $(\mu, \beta)$

The same animation as in §5.2, but for the $(\mu, \beta)$ sampler. Each Gibbs iteration is a **horizontal** move (update $\mu$ at fixed $\beta$) followed by a **vertical** move (update $\beta$ at fixed $\mu$).

- The first **20 sub-steps** play at **5 sub-steps per second**.
- After that, the chain plays at **5× speed**.

Compare to §5.2: instead of a thin ridge, each sub-step lands somewhere in a roundish region, and the chain quickly fills out the posterior.

In [ ]:
# Build animation showing each conditional sub-step in (mu, beta) space.

n_slow_mb = 20
slow_frames_per_step_mb = 5
fast_steps_per_frame_mb = 5
n_fast_frames_mb = 250

frame_to_idx_mb = []
for k in range(1, n_slow_mb + 1):
    for _ in range(slow_frames_per_step_mb):
        frame_to_idx_mb.append(k)
current = n_slow_mb
max_idx_mb = len(mu_sub) - 1
for _ in range(n_fast_frames_mb):
    current = min(current + fast_steps_per_frame_mb, max_idx_mb)
    frame_to_idx_mb.append(current)
    if current == max_idx_mb:
        break

n_frames_mb = len(frame_to_idx_mb)
last_idx_mb = frame_to_idx_mb[-1]

fig_b, ax_b = plt.subplots(figsize=(7, 6))
ax_b.set_xlim(mu_sub[:last_idx_mb + 1].min()  - 0.1, mu_sub[:last_idx_mb + 1].max()  + 0.1)
ax_b.set_ylim(beta_mb_sub[:last_idx_mb + 1].min() - 1.0, beta_mb_sub[:last_idx_mb + 1].max() + 1.0)
ax_b.set_xlabel(r'$\mu$')
ax_b.set_ylabel(r'$\beta$')
ax_b.set_title(r'Gibbs in $(\mu, \beta)$: alternating $\mu$ (horizontal) and $\beta$ (vertical) updates')

line_b, = ax_b.plot([], [], color=COLOR_MUBETA, linewidth=0.7, alpha=0.6)
head_b, = ax_b.plot([], [], 'o', color='black', markersize=10)
text_b  = ax_b.text(0.02, 0.97, '', transform=ax_b.transAxes, va='top', fontsize=11,
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

slow_frames_total_mb = n_slow_mb * slow_frames_per_step_mb

def init_b():
    line_b.set_data([], [])
    head_b.set_data([], [])
    text_b.set_text('')
    return line_b, head_b, text_b

def update_b(k):
    end = frame_to_idx_mb[k]
    line_b.set_data(mu_sub[:end + 1], beta_mb_sub[:end + 1])
    head_b.set_data([mu_sub[end]], [beta_mb_sub[end]])
    iter_num = (end + 1) // 2
    if end == 0:
        ut = 'initial state'
    elif end % 2 == 1:
        ut = r'just updated $\mu$ (horizontal)'
    else:
        ut = r'just updated $\beta$ (vertical)'
    speed = 'slow' if k < slow_frames_total_mb else '5x speed'
    text_b.set_text(f'iter {iter_num}: {ut}  [{speed}]')
    return line_b, head_b, text_b

anim_mb = animation.FuncAnimation(
    fig_b, update_b, frames=n_frames_mb,
    init_func=init_b, blit=True, interval=40
)
plt.close(fig_b)
HTML(anim_mb.to_jshtml())

---

## 8. Comparison: same posterior?

A quick sanity check: both samplers should target the same posterior. Let's compare the posterior-mean estimates of $\alpha$ and $\beta$ produced by the two chains we just ran (each $4{,}500$ samples after the burn-in of $500$).

In [ ]:
print(f"After {n_iter} iterations ({n_iter - burn} samples post burn-in):")
print()
print(f"  (alpha, beta) sampler:  E[alpha] = {alpha_tr[burn:].mean():6.2f}     E[beta] = {beta_tr[burn:].mean():6.2f}")
print(f"  (mu, beta)    sampler:  E[alpha] = {alpha_from_mubeta[burn:].mean():6.2f}     E[beta] = {beta_mb_tr[burn:].mean():6.2f}")

Hmm — those numbers don't quite agree. The $(\alpha, \beta)$ sampler reports a noticeably higher posterior mean for $\alpha$ (and $\beta$). **Are the two samplers actually targeting different posteriors?**

Most likely: no. The $(\alpha, \beta)$ chain mixes slowly, so $4{,}500$ samples just isn't enough to nail down the posterior mean. The natural check: **run the chains $10\times$ longer and see whether the estimates converge.**

In [ ]:
# Re-run both samplers for 10x as many iterations
n_iter_long = 10 * n_iter
alpha_sub_L, beta_sub_L, _ = run_alphabeta_sampler(n_iter_long)
alpha_tr_L = alpha_sub_L[2::2]
beta_tr_L  = beta_sub_L[2::2]

mu_sub_L, beta_mb_sub_L, _ = run_mubeta_sampler(n_iter_long)
mu_tr_L      = mu_sub_L[2::2]
beta_mb_tr_L = beta_mb_sub_L[2::2]
alpha_from_mubeta_L = mu_tr_L * beta_mb_tr_L

print(f"After {n_iter_long} iterations ({n_iter_long - burn} samples post burn-in):")
print()
print(f"  (alpha, beta) sampler:  E[alpha] = {alpha_tr_L[burn:].mean():6.2f}     E[beta] = {beta_tr_L[burn:].mean():6.2f}")
print(f"  (mu, beta)    sampler:  E[alpha] = {alpha_from_mubeta_L[burn:].mean():6.2f}     E[beta] = {beta_mb_tr_L[burn:].mean():6.2f}")

Now the posterior means agree to within a fraction of a unit. So both chains *are* sampling the same posterior; the $(\alpha, \beta)$ chain just needs more iterations to settle down.

To see this convergence in action, plot the **running** posterior-mean estimate of $E[\beta \mid X]$ as a function of how many iterations have been collected so far.

In [ ]:
# Running posterior-mean estimate of E[beta | data] over iterations
running_mean_ab = np.cumsum(beta_tr_L[burn:])      / np.arange(1, n_iter_long - burn + 1)
running_mean_mb = np.cumsum(beta_mb_tr_L[burn:])   / np.arange(1, n_iter_long - burn + 1)
target = running_mean_mb[-1]   # essentially the converged value
iters  = np.arange(burn + 1, n_iter_long + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(iters, running_mean_ab, color=COLOR_AB,     linewidth=1.5,
        label=r'$(\alpha, \beta)$ sampler')
ax.plot(iters, running_mean_mb, color=COLOR_MUBETA, linewidth=1.5,
        label=r'$(\mu, \beta)$ sampler')
ax.axhline(target, color='black', linewidth=0.8, linestyle='--', alpha=0.6,
           label=fr'apparent limit $\approx$ {target:.2f}')
ax.set_xscale('log')
ax.set_xlabel('iteration (log scale)')
ax.set_ylabel(r'running estimate of $E[\beta \mid X]$')
ax.set_title(r'How fast does each chain home in on the posterior mean of $\beta$?')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

*Both running averages eventually converge to the same value, but the $(\mu, \beta)$ sampler (magenta) settles within a few hundred iterations, while the $(\alpha, \beta)$ sampler (blue) is still wandering at $5{,}000$ iterations and only stabilizes much later. Same posterior, vastly different convergence rates.*

### 8.1 Quantifying the difference: effective sample size

By eye, the $(\mu, \beta)$ sampler is clearly faster. To put a number on this we use the **effective sample size (ESS)**.

If our chain produced $T$ truly independent draws, the standard error of any posterior-mean estimate would be $\sigma/\sqrt{T}$. But MCMC draws are correlated — successive iterations share information — so a chain of length $T$ delivers *less* than $T$ independent samples' worth of information. The ESS is the size of an iid sample that would have *the same Monte Carlo variance* as our correlated chain:

$$\text{ESS} \;=\; \frac{T}{1 + 2\sum_{k \geq 1} \rho_k},$$

where $\rho_k$ is the lag-$k$ autocorrelation of the chain. For an iid sample, $\rho_k = 0$ for all $k \geq 1$ and $\text{ESS} = T$. For a slow-mixing chain, $\rho_k$ decays slowly, the sum is large, and $\text{ESS} \ll T$.

Below we compute the autocorrelation of $\alpha$ in each chain, then plug into the ESS formula. (The sum is truncated at the first negative crossing — Geyer's initial positive sequence — to keep the estimate well-behaved.)

In [ ]:
def autocorr(x, max_lag):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    c0 = np.dot(x, x) / len(x)
    return np.array([np.dot(x[:len(x)-k], x[k:]) / (len(x)-k) / c0
                     for k in range(max_lag + 1)])

def ess(acf, N):
    pos = acf[1:].copy()
    cutoff = np.argmax(pos < 0) if np.any(pos < 0) else len(pos)
    return N / (1 + 2 * pos[:cutoff].sum())

max_lag = 1000
acf_ab = autocorr(alpha_tr_L[burn:],         max_lag)
acf_mb = autocorr(alpha_from_mubeta_L[burn:], max_lag)

N_post = n_iter_long - burn
ess_ab = ess(acf_ab, N_post)
ess_mb = ess(acf_mb, N_post)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(acf_ab, color=COLOR_AB,     linewidth=2,
        label=fr'$(\alpha, \beta)$ sampler:  ESS for $\alpha$ $\approx$ {ess_ab:.0f} of {N_post}')
ax.plot(acf_mb, color=COLOR_MUBETA, linewidth=2,
        label=fr'$(\mu, \beta)$ sampler:  ESS for $\alpha$ $\approx$ {ess_mb:.0f} of {N_post}')
ax.axhline(0, color='k', linewidth=0.5)
ax.set_xlabel('lag')
ax.set_ylabel(r'autocorrelation of $\alpha$ samples')
ax.set_title('Autocorrelation of $\\alpha$ in the two parametrizations')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f"Effective sample size for alpha:")
print(f"  (alpha, beta) sampler:  ESS = {ess_ab:6.0f}  ({100*ess_ab/N_post:.1f}% of {N_post} draws)")
print(f"  (mu, beta)    sampler:  ESS = {ess_mb:6.0f}  ({100*ess_mb/N_post:.1f}% of {N_post} draws)")
print(f"  Speedup factor:        {ess_mb / ess_ab:.1f}x")

### 8.2 Side-by-side traces in $(\alpha, \beta)$ space (long chains)

In [ ]:
beta_from_mubeta_L = beta_mb_tr_L
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

axes[0].plot(alpha_tr_L[burn:], beta_tr_L[burn:],
             color=COLOR_AB, linewidth=0.2, alpha=0.4)
axes[0].scatter(alpha_tr_L[burn:], beta_tr_L[burn:],
                color=COLOR_AB, s=2, alpha=0.25)
axes[0].set_xlabel(r'$\alpha$')
axes[0].set_ylabel(r'$\beta$')
axes[0].set_title(r'Direct $(\alpha, \beta)$ sampler')

axes[1].plot(alpha_from_mubeta_L[burn:], beta_from_mubeta_L[burn:],
             color=COLOR_MUBETA, linewidth=0.2, alpha=0.4)
axes[1].scatter(alpha_from_mubeta_L[burn:], beta_from_mubeta_L[burn:],
                color=COLOR_MUBETA, s=2, alpha=0.25)
axes[1].set_xlabel(r'$\alpha$')
axes[1].set_title(r'$(\mu, \beta)$ sampler, transformed to $(\alpha, \beta)$')

for ax in axes:
    ax.set_xlim(min(alpha_tr_L[burn:].min(), alpha_from_mubeta_L[burn:].min()) - 1,
                max(alpha_tr_L[burn:].max(), alpha_from_mubeta_L[burn:].max()) + 1)
    ax.set_ylim(min(beta_tr_L[burn:].min(),  beta_from_mubeta_L[burn:].min())  - 0.5,
                max(beta_tr_L[burn:].max(),  beta_from_mubeta_L[burn:].max())  + 0.5)
plt.tight_layout()
plt.show()

*With $50{,}000$ iterations, both clouds now occupy essentially the same region in $(\alpha, \beta)$ space — confirming they sample the same posterior. The right cloud (the $(\mu, \beta)$ sampler transformed back) was already filled in within a few thousand iterations; the left cloud (the $(\alpha, \beta)$ sampler) needed all $50{,}000$ to spread out comparably.*

---

## 9. Why does reparametrization help so much?

The Gibbs sampler updates one coordinate at a time. After an update of $\beta$ (with $\alpha$ fixed), the new state is "as far as a single $\beta$ step allows" — but the size of that step is constrained by the conditional distribution of $\beta$ *given* $\alpha$.

In the $(\alpha, \beta)$ parametrization, the posterior is concentrated on a narrow ridge $\alpha/\beta \approx \mu_{\text{true}}$. So the conditional of $\beta$ given $\alpha$ is a *very* narrow distribution centered near $\alpha/\mu_{\text{true}}$. Each $\beta$ update can only nudge by that small amount; same for each $\alpha$ update. Net effect: tiny zig-zag steps along the ridge.

In the $(\mu, \beta)$ parametrization, the posterior is roughly axis-aligned — $\mu$ and $\beta$ are essentially independent. Each conditional has full-width support, and a single update can cross the bulk of the posterior. Far fewer iterations are needed for an effectively independent draw.

**General principle.** Gibbs samplers mix slowly when the parameters are correlated in the posterior. Reparametrizing to (approximately) independent coordinates is one of the most reliable ways to speed up MCMC.

### Lab takeaways

1. **Conjugate full conditionals are easy.** In the $(\alpha, \beta)$ parametrization, the $\theta_i$ and $\beta$ updates are one-line Gamma draws.
2. **Non-conjugate full conditionals are still easy in 1D.** Compute $\log p$ on a grid, exponentiate, normalize, sample. Works for any one-dimensional density we can evaluate. We used this once for $\alpha$ in §4, then for both $\mu$ and $\beta$ in §7 (where the transformed prior breaks Gamma-Gamma conjugacy — but it doesn't matter, the 1D trick still works).
3. **Parameterization matters.** Two valid parameterizations of the *same* posterior can give wildly different mixing speeds. The $(\mu, \beta)$ parametrization here gives roughly **20× the effective sample size** for $\alpha$ at zero extra cost.
4. **Diagnose mixing with autocorrelation / ESS.** A 2D trace plot also tells the story at a glance: a thin ridge means slow mixing; a roundish cloud means fast.